# القدرة على التنبؤ الفوري (Real-Time Prediction Capability) — D-MorphNet

هذا الدفتر ينفّذ قسم **"Real-Time Prediction Capability"**: الهدف الرئيسي هو أن يحدد النظام **بسرعة** هل الصورة حقيقية أم مدموجة — ليس فقط على بيانات الاختبار، بل على **صور جديدة تُرفع مباشرة** (كما في الشكلين ٤ و٥).

## مسار التنبؤ الفوري (كما في الورقة)
1. **رفع الصورة** إلى النظام.
2. **المعالجة المسبقة والتسوية** (تغيير الحجم إلى 528×528 + CLAHE — نفس خطوات التدريب تماماً).
3. **استخراج السمات** بـ EfficientNet-B6.
4. تمرير السمات إلى **SVM** للتصنيف.
5. عرض القرار فوراً: **MORPH IMAGE** أو **REAL IMAGE** مع درجة الثقة (Confidence) — بنفس أسلوب عرض الشكلين ٤ و٥.

النظام يعمل على **الصور أحادية الوجه والمتعددة الوجوه**: في حالة تعدد الوجوه يكتشفها جميعاً ويصنّف كل وجه على حدة.

> ⚙️ يفضَّل GPU لتسريع استخراج السمات، ويحتاج الدفتر النماذج المحفوظة في Drive من الدفاتر السابقة.


## ١) تجهيز البيئة


In [ ]:
!pip -q install mediapipe opencv-python-headless scikit-learn joblib

import os, glob, time, random, shutil
import numpy as np
import cv2
import tensorflow as tf
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

gpus = tf.config.list_physical_devices('GPU')
print("GPU:", gpus if gpus else "⚠️ لا يوجد GPU — سيعمل النظام لكن أبطأ")


## ٢) تحميل مكوّنات النظام المدرّبة

نحمّل من Google Drive كل ما دُرّب في الدفاتر السابقة:
- نموذج السمات **EfficientNet-B6** المضبوط دقيقاً (أو B6 مجمّد كبديل).
- **الـ Scaler** ومصنّف **SVM** النهائي.
- **العتبة المثلى** المختارة في دفتر تحسين العتبة (وإلا الافتراضية 0).


In [ ]:
import joblib

DRIVE_DIR = "/content/drive/MyDrive/DMorphNet_models"
if not os.path.isdir(DRIVE_DIR):
    from google.colab import drive
    drive.mount('/content/drive')

IMG_SIZE = 528
from tensorflow.keras.applications.efficientnet import preprocess_input

# 1) نموذج السمات
feat_path = os.path.join(DRIVE_DIR, "effb6_finetuned_features.keras")
if os.path.exists(feat_path):
    feat_model = tf.keras.models.load_model(feat_path)
    print("✅ تم تحميل نموذج B6 المضبوط دقيقاً")
else:
    from tensorflow.keras.applications import EfficientNetB6
    feat_model = EfficientNetB6(include_top=False, weights="imagenet",
                                pooling="avg",
                                input_shape=(IMG_SIZE, IMG_SIZE, 3))
    print("⚠️ تم بناء B6 مجمّد (لم يُعثر على النموذج المضبوط)")

# 2) المصنّف والـ Scaler
svm_final = joblib.load(os.path.join(DRIVE_DIR, "svm_hybrid_final.joblib"))
scaler = joblib.load(os.path.join(DRIVE_DIR, "scaler_hybrid_final.joblib"))
print("✅ تم تحميل SVM والـ Scaler")

# 3) العتبة المثلى
tau_path = os.path.join(DRIVE_DIR, "optimal_threshold.npz")
TAU = float(np.load(tau_path)["tau"]) if os.path.exists(tau_path) else 0.0
print(f"عتبة القرار المستخدمة: τ = {TAU:.4f}")


## ٣) دوال المعالجة والتنبؤ لوجه واحد

- **المعالجة**: نفس خطوات التدريب حرفياً (528×528 + CLAHE على قناة L) — أي اختلاف هنا يفسد التنبؤ.
- **الثقة (Confidence)**: نحوّل درجة القرار $s = w^TF+b$ إلى قيمة بين 0 و1 بدالة سيغمويد $\text{conf} = \sigma(s - \tau)$:
  - قيمة قريبة من **1.0** ← مدموجة بثقة عالية (كما في الشكل ٤: Confidence 1.0000).
  - قيمة قريبة من **0.0** ← حقيقية بثقة عالية (كما في الشكل ٥: Confidence 0.0081).


In [ ]:
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))


def preprocess_face(img_bgr):
    # نفس معالجة التدريب: 528x528 + CLAHE على قناة الإضاءة
    img = cv2.resize(img_bgr, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_CUBIC)
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = clahe.apply(l)
    return cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2BGR)


def predict_face(img_bgr):
    # المسار الكامل لوجه واحد — ترجع (القرار، الثقة، درجة القرار، الأزمنة)
    t0 = time.perf_counter()
    proc = preprocess_face(img_bgr)
    rgb = cv2.cvtColor(proc, cv2.COLOR_BGR2RGB).astype(np.float32)
    t1 = time.perf_counter()
    feat = feat_model.predict(preprocess_input(rgb)[None, ...], verbose=0)
    t2 = time.perf_counter()
    s = float(svm_final.decision_function(scaler.transform(feat))[0])
    is_morph = s >= TAU
    confidence = 1.0 / (1.0 + np.exp(-(s - TAU)))     # سيغمويد حول العتبة
    t3 = time.perf_counter()
    times = {"preprocess": t1 - t0, "features": t2 - t1, "svm": t3 - t2,
             "total": t3 - t0}
    return ("MORPH IMAGE" if is_morph else "REAL IMAGE"), confidence, s, times


def show_result(img_bgr, label, confidence):
    # عرض النتيجة بنفس أسلوب الشكلين 4 و5: عنوان أحمر فوق الصورة
    plt.figure(figsize=(5, 6))
    plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    plt.title(f"{label}\nConfidence:{confidence:.4f}",
              color="red", fontsize=15, fontweight="bold")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

print("✅ دوال التنبؤ الفوري جاهزة")


## ٤) دعم الصور متعددة الوجوه

كما تذكر الورقة، النظام يعمل على الصور أحادية **ومتعددة** الوجوه:
1. نكتشف كل الوجوه في الصورة بـ MediaPipe Face Detection.
2. نقصّ كل وجه (مع هامش 25%) ونمرّره في نفس مسار التنبؤ.
3. نرسم مربعاً حول كل وجه: **أحمر = مدموج، أخضر = حقيقي** مع الثقة.


In [ ]:
import mediapipe as mp

# كاشف وجوه موحد يعمل مع نسختي MediaPipe (القديمة solutions والجديدة Tasks)
if hasattr(mp, "solutions"):
    _detector = mp.solutions.face_detection.FaceDetection(
        model_selection=1, min_detection_confidence=0.5)

    def _raw_face_boxes(img_bgr):
        h, w = img_bgr.shape[:2]
        res = _detector.process(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
        if not res.detections:
            return []
        out = []
        for det in res.detections:
            bb = det.location_data.relative_bounding_box
            out.append((int(bb.xmin * w), int(bb.ymin * h),
                        int(bb.width * w), int(bb.height * h)))
        return out
else:
    import urllib.request
    from mediapipe.tasks.python import BaseOptions
    from mediapipe.tasks.python import vision as mp_vision
    _DET_MODEL = "/content/blaze_face_short_range.tflite"
    if not os.path.exists(_DET_MODEL):
        urllib.request.urlretrieve(
            "https://storage.googleapis.com/mediapipe-models/face_detector/"
            "blaze_face_short_range/float16/1/blaze_face_short_range.tflite",
            _DET_MODEL)
    _detector = mp_vision.FaceDetector.create_from_options(
        mp_vision.FaceDetectorOptions(
            base_options=BaseOptions(model_asset_path=_DET_MODEL),
            min_detection_confidence=0.5))

    def _raw_face_boxes(img_bgr):
        rgb = np.ascontiguousarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        res = _detector.detect(mp_img)
        return [(int(d.bounding_box.origin_x), int(d.bounding_box.origin_y),
                 int(d.bounding_box.width), int(d.bounding_box.height))
                for d in res.detections]


def detect_faces(img_bgr):
    # ترجع قائمة مربعات (x, y, w, h) لكل وجه مكتشف مع هامش 25%
    h, w = img_bgr.shape[:2]
    boxes = []
    for (x, y, bw, bh) in _raw_face_boxes(img_bgr):
        m = int(0.25 * max(bw, bh))
        x0, y0 = max(0, x - m), max(0, y - m)
        x1, y1 = min(w, x + bw + m), min(h, y + bh + m)
        boxes.append((x0, y0, x1 - x0, y1 - y0))
    return boxes


def predict_image(img_bgr):
    # يصنّف كل وجه في الصورة؛ وإن لم يُكتشف وجه يصنّف الصورة كاملة
    boxes = detect_faces(img_bgr)
    annotated = img_bgr.copy()
    results = []
    if not boxes:
        label, conf, s, times = predict_face(img_bgr)
        return [(None, label, conf, times)], annotated

    for (x, y, bw, bh) in boxes:
        crop = img_bgr[y:y + bh, x:x + bw]
        label, conf, s, times = predict_face(crop)
        results.append(((x, y, bw, bh), label, conf, times))
        color = (0, 0, 255) if label == "MORPH IMAGE" else (0, 200, 0)
        cv2.rectangle(annotated, (x, y), (x + bw, y + bh), color, 3)
        cv2.putText(annotated, f"{label.split()[0]} {conf:.2f}",
                    (x, max(25, y - 8)), cv2.FONT_HERSHEY_SIMPLEX,
                    0.8, color, 2)
    return results, annotated

print("✅ دعم تعدد الوجوه جاهز")


## ٥) التنبؤ الفوري: ارفع صوراً جديدة الآن 📤

شغّل الخلية واختر أي صورة (أو عدة صور) من جهازك — تماماً كما في الشكلين ٤ و٥:
- صورة بوجه واحد ← تُعرض بعنوان أحمر **MORPH IMAGE** أو **REAL IMAGE** مع الثقة.
- صورة بعدة وجوه ← تُعرض مع مربع ملوّن ونتيجة لكل وجه.


In [ ]:
from google.colab import files

uploaded = files.upload()

for fname in uploaded:
    img = cv2.imdecode(np.frombuffer(uploaded[fname], np.uint8),
                       cv2.IMREAD_COLOR)
    if img is None:
        print(f"⚠️ تعذّرت قراءة {fname}")
        continue

    results, annotated = predict_image(img)

    if len(results) == 1 and results[0][0] is None:
        # وجه واحد / صورة كاملة — عرض بأسلوب الشكلين 4 و5
        _, label, conf, times = results[0]
        show_result(img, label, conf)
    else:
        plt.figure(figsize=(8, 8))
        plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        plt.title(f"عدد الوجوه المكتشفة: {len(results)}", fontsize=13)
        plt.axis("off")
        plt.tight_layout()
        plt.show()

    for i, (box, label, conf, times) in enumerate(results, 1):
        print(f"  وجه {i}: {label}  Confidence:{conf:.4f}  "
              f"(زمن التنبؤ {times['total']*1000:.0f} ms)")


## ٦) تجربة سريعة على عيّنات من مجموعة الاختبار

إن لم تكن لديك صور لرفعها، تعرض هذه الخلية النتيجة بنفس أسلوب الشكلين ٤ و٥ على عيّنة حقيقية وعيّنة مدموجة من مجموعة الاختبار.


In [ ]:
PROC_DIR = "/content/DMorphNet_processed"
if not os.path.isdir(os.path.join(PROC_DIR, "test")):
    zip_path = "/content/drive/MyDrive/DMorphNet_processed.zip"
    if os.path.exists(zip_path):
        shutil.unpack_archive(zip_path, PROC_DIR)

if os.path.isdir(os.path.join(PROC_DIR, "test")):
    for cls in ["real", "morph"]:
        p = random.choice(glob.glob(os.path.join(PROC_DIR, "test", cls, "*.jpg")))
        img = cv2.imread(p)
        label, conf, s, times = predict_face(img)
        truth = "حقيقية" if cls == "real" else "مدموجة"
        print(f"الحقيقة: {truth} — التنبؤ: {label} — "
              f"Confidence:{conf:.4f} — {times['total']*1000:.0f} ms")
        show_result(img, label, conf)
else:
    print("مجموعة الاختبار غير متوفرة — استخدم خلية الرفع أعلاه")


## ٧) قياس سرعة التنبؤ (زمن كل مرحلة)

"بعد استخراج السمات، يعطي نظامنا التنبؤ بسرعة كبيرة" — نقيس ذلك عملياً: متوسط زمن كل مرحلة على 20 تنبؤاً (بعد تنبؤ إحماء أول لتهيئة GPU):
- المعالجة المسبقة (تغيير حجم + CLAHE).
- استخراج السمات بـ B6 (الأثقل).
- قرار SVM (شبه لحظي).


In [ ]:
sample_paths = glob.glob(os.path.join(PROC_DIR, "test", "*", "*.jpg"))
if sample_paths:
    warm = cv2.imread(sample_paths[0])
    predict_face(warm)                     # إحماء (تهيئة GPU وتجميع الدوال)

    N_RUNS = 20
    agg = {"preprocess": [], "features": [], "svm": [], "total": []}
    for p in random.sample(sample_paths, min(N_RUNS, len(sample_paths))):
        img = cv2.imread(p)
        _, _, _, times = predict_face(img)
        for k in agg:
            agg[k].append(times[k])

    print(f"متوسط الأزمنة على {N_RUNS} تنبؤاً:")
    for k, name in [("preprocess", "المعالجة المسبقة"),
                    ("features", "استخراج السمات (B6)"),
                    ("svm", "تصنيف SVM"),
                    ("total", "الإجمالي")]:
        print(f"  {name:22s}: {np.mean(agg[k])*1000:7.1f} ms")

    fps = 1.0 / np.mean(agg["total"])
    print(f"\nمعدل المعالجة: ~{fps:.1f} صورة/ثانية — "
          "قرار شبه فوري بعد استخراج السمات، كما تذكر الورقة")
else:
    print("لا توجد صور متاحة للقياس — استخدم خلية الرفع أولاً")


## ٨) الخلاصة

اكتمل النظام الفوري بالكامل:
- **رفع صورة جديدة** ← معالجة موحّدة ← سمات B6 ← قرار SVM ← عرض **MORPH IMAGE / REAL IMAGE** مع الثقة (الشكلان ٤ و٥).
- يدعم **وجهاً واحداً أو عدة وجوه** في الصورة نفسها.
- القرار بعد استخراج السمات **شبه لحظي** (SVM أجزاء من الميلي ثانية)، والزمن الكلي لكل صورة يظهر في خلية القياس أعلاه.

بهذا تكتمل سلسلة دفاتر D-MorphNet من بناء البيانات حتى النشر الفوري. 🎉


In [ ]:
print("مكوّنات النظام الفوري المستخدمة:")
print(f"  نموذج السمات : EfficientNet-B6 ({'مضبوط دقيقاً' if os.path.exists(feat_path) else 'مجمّد'})")
print(f"  المصنّف      : SVM (τ = {TAU:.4f})")
print(f"  حجم الإدخال  : {IMG_SIZE}x{IMG_SIZE} + CLAHE")
print("  الوضعان      : وجه واحد / عدة وجوه")
print("\n🎉 نظام D-MorphNet جاهز للاستخدام الفوري")
